In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import(
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    VotingClassifier
)
from sklearn.metrics import(
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [2]:
df=pd.read_csv("data\loan_approval_dataset.csv")
df.columns=df.columns.str.strip()

<>:1: SyntaxWarning: invalid escape sequence '\l'
<>:1: SyntaxWarning: invalid escape sequence '\l'
C:\Users\sukal\AppData\Local\Temp\ipykernel_21160\3627673541.py:1: SyntaxWarning: invalid escape sequence '\l'
  df=pd.read_csv("data\loan_approval_dataset.csv")


In [3]:
print(df.head())
print(df.info())

   loan_id  no_of_dependents      education self_employed  income_annum  \
0        1                 2       Graduate            No       9600000   
1        2                 0   Not Graduate           Yes       4100000   
2        3                 3       Graduate            No       9100000   
3        4                 3       Graduate            No       8200000   
4        5                 5   Not Graduate           Yes       9800000   

   loan_amount  loan_term  cibil_score  residential_assets_value  \
0     29900000         12          778                   2400000   
1     12200000          8          417                   2700000   
2     29700000         20          506                   7100000   
3     30700000          8          467                  18200000   
4     24200000         20          382                  12400000   

   commercial_assets_value  luxury_assets_value  bank_asset_value loan_status  
0                 17600000             22700000           80

In [4]:
df.drop("loan_id",axis=1,inplace=True)

In [5]:
print("\nMissing Values\n")
print(df.isnull().sum())
for col in df.columns:
    if df[col].dtype=="object":
        df[col].fillna(df[col].mode()[0])
    else:
        df[col].fillna(df[col].median())


Missing Values

no_of_dependents            0
education                   0
self_employed               0
income_annum                0
loan_amount                 0
loan_term                   0
cibil_score                 0
residential_assets_value    0
commercial_assets_value     0
luxury_assets_value         0
bank_asset_value            0
loan_status                 0
dtype: int64


In [15]:
encoders={}
categorical_cols=[
    "education",
    "self_employed",
    "loan_status"
]
for col in categorical_cols:
    encoder=LabelEncoder()
    df[col]=encoder.fit_transform(df[col])
    encoders[col]=encoder
joblib.dump(encoders,"label_encoders.pkl")
print(df.head())

   no_of_dependents  education  self_employed  income_annum  loan_amount  \
0                 2          0              0       9600000     29900000   
1                 0          1              1       4100000     12200000   
2                 3          0              0       9100000     29700000   
3                 3          0              0       8200000     30700000   
4                 5          1              1       9800000     24200000   

   loan_term  cibil_score  residential_assets_value  commercial_assets_value  \
0         12          778                   2400000                 17600000   
1          8          417                   2700000                  2200000   
2         20          506                   7100000                  4500000   
3          8          467                  18200000                  3300000   
4         20          382                  12400000                  8200000   

   luxury_assets_value  bank_asset_value  loan_status  
0     

In [7]:
x=df.drop("loan_status",axis=1)
y=df["loan_status"]

In [8]:
scaler=StandardScaler()
x=scaler.fit_transform(x)
joblib.dump(scaler,"scaler.pkl")

['scaler.pkl']

In [9]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
models = {
    "Logistic Regression":
        LogisticRegression(max_iter=1000),
    "KNN":
        KNeighborsClassifier(n_neighbors=5),
    "SVM":
        SVC(probability=True),
    "Decision Tree":
        DecisionTreeClassifier(random_state=42),
    "Random Forest":
        RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ),
    "Gradient Boosting":
        GradientBoostingClassifier(random_state=42),
    "AdaBoost":
        AdaBoostClassifier(random_state=42)
}

In [11]:
results = []
best_model = None
best_accuracy = 0
print("\n================ MODEL RESULTS ================\n")
for name, model in models.items():
    print("=" * 60)
    print(name)
    model.fit(x_train, y_train)
    prediction = model.predict(x_test)
    accuracy = accuracy_score(y_test, prediction)
    print(f"Accuracy : {accuracy:.4f}")
    print("\nClassification Report")
    print(classification_report(y_test, prediction))
    print("Confusion Matrix")
    print(confusion_matrix(y_test, prediction))
    results.append([name, accuracy])
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = model


================ MODEL RESULTS ================

Logistic Regression
Accuracy : 0.9227

Classification Report
              precision    recall  f1-score   support

           0       0.93      0.95      0.94       531
           1       0.92      0.88      0.90       323

    accuracy                           0.92       854
   macro avg       0.92      0.91      0.92       854
weighted avg       0.92      0.92      0.92       854

Confusion Matrix
[[505  26]
 [ 40 283]]
KNN
Accuracy : 0.8970

Classification Report
              precision    recall  f1-score   support

           0       0.91      0.93      0.92       531
           1       0.88      0.85      0.86       323

    accuracy                           0.90       854
   macro avg       0.89      0.89      0.89       854
weighted avg       0.90      0.90      0.90       854

Confusion Matrix
[[492  39]
 [ 49 274]]
SVM
Accuracy : 0.9450

Classification Report
              precision    recall  f1-score   support

          

In [12]:
voting = VotingClassifier(
    estimators=[
        ("lr", models["Logistic Regression"]),
        ("knn", models["KNN"]),
        ("svm", models["SVM"]),
        ("dt", models["Decision Tree"]),
        ("rf", models["Random Forest"]),
        ("gb", models["Gradient Boosting"]),
        ("ab", models["AdaBoost"])
    ],
    voting="hard"
)
voting.fit(x_train, y_train)
prediction = voting.predict(x_test)
accuracy = accuracy_score(y_test, prediction)
print("=" * 60)
print("Voting Classifier")
print(f"Accuracy : {accuracy:.4f}")
print("\nClassification Report")
print(classification_report(y_test, prediction))
print("Confusion Matrix")
print(confusion_matrix(y_test, prediction))
results.append(["Voting Classifier", accuracy])
if accuracy > best_accuracy:
    best_accuracy = accuracy
    best_model = voting

Voting Classifier
Accuracy : 0.9778

Classification Report
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       531
           1       0.97      0.97      0.97       323

    accuracy                           0.98       854
   macro avg       0.98      0.98      0.98       854
weighted avg       0.98      0.98      0.98       854

Confusion Matrix
[[523   8]
 [ 11 312]]


In [13]:
results_df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy"]
)
results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)
print("\n================ Accuracy Comparison ================\n")
print(results_df)


================ Accuracy Comparison ================

                 Model  Accuracy
4        Random Forest  0.983607
5    Gradient Boosting  0.982436
7    Voting Classifier  0.977752
6             AdaBoost  0.974239
3        Decision Tree  0.971897
2                  SVM  0.944965
0  Logistic Regression  0.922717
1                  KNN  0.896956


In [14]:
joblib.dump(best_model, "model.pkl")
print("Best Model Saved Successfully!")
print(best_model)
print(f"Best Accuracy : {best_accuracy:.4f}")

Best Model Saved Successfully!
RandomForestClassifier(random_state=42)
Best Accuracy : 0.9836
